# Urban Energy Coordination with Temporal Sparsity Control

This notebook is the **clean, GitHub-ready reference implementation** used for the main validation experiment of the paper.

## Title of the paper
**Temporal-Sparsity-Aware Urban Energy Coordination for Peak Reduction, Excess Mitigation, and Storage-Augmented Demand Management**

## What this notebook does
We use a **single public dataset** (`summer_peaks.csv`) as the main experimental basis in order to keep the study coherent, reproducible, and easy to justify. The workflow combines:
- demand prediction at building level,
- district-level aggregation,
- storage-augmented coordination,
- thresholded overload control,
- budgeted activation of significant violation periods.

## Problem
Purely convex coordination formulations can reduce peak and excess demand while still leaving many small residual overloads spread over time. If evaluation relies only on raw overload counts, the interpretation may be misleading.

## Our solution
The notebook therefore focuses on five core ideas:
- **peak reduction**
- **excess reduction**
- **temporal sparsity control**
- **thresholded violation metric**
- **budgeted activation constraint**

The primary stress indicator in this notebook is the **thresholded violation-day metric**, not the raw overload-day count. This choice reflects the main scientific lesson from the broader experimentation process: physically meaningful overload persistence should be measured using a thresholded notion of significant violation.

## Why only one dataset here
This final notebook intentionally uses **Dataset 1 only**. The goal is to provide a clean and defensible validation pipeline on a single public benchmark-style dataset, rather than mixing structurally different datasets in one main experiment.

## Outputs produced
The notebook generates:
- paper-ready tables,
- paper-ready figures,
- a rich `outputs_summary.txt`,
- and a compact ablation study to show which modeling components matter most.


## Notebook roadmap

Each Python cell is preceded by a short text cell so that a reader can understand the logic quickly on GitHub or in Colab.

**Workflow**
1. setup and output paths  
2. data loading and profiling  
3. preprocessing and synthetic district construction  
4. prediction model training  
5. district aggregation  
6. optimization configuration and solver definition  
7. main optimization run  
8. paper-ready tables and figures  
9. ablation study  
10. rich summary export

**Design choice**
This notebook keeps the narrative minimal and practical. It does not include LaTeX methodology text; instead, the implementation is explained through short section notes and interpretable outputs.


## Setup and output folders

This cell imports the required packages, mounts Google Drive in Colab, defines input/output paths, and creates a few helper utilities.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, explained_variance_score

SEED = 42
np.random.seed(SEED)

DATASET_PATH = "/content/drive/MyDrive/Datasets/Energy/summer_peaks.csv"
OUTPUT_BASE_DIR = "/content/drive/MyDrive/Outputs"
OUTPUT_FIGURES_DIR = os.path.join(OUTPUT_BASE_DIR, "figures")
OUTPUT_TABLES_DIR = os.path.join(OUTPUT_BASE_DIR, "tables")
SUMMARY_PATH = os.path.join(OUTPUT_BASE_DIR, "outputs_summary.txt")

Path(OUTPUT_FIGURES_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_TABLES_DIR).mkdir(parents=True, exist_ok=True)

def resolve_dataset_path(default_path):
    if os.path.exists(default_path):
        return default_path
    search_root = Path("/content/drive/MyDrive")
    candidates = list(search_root.rglob("summer_peaks.csv"))
    if not candidates:
        raise FileNotFoundError(f"Dataset not found: {default_path}")
    return str(candidates[0])

def save_table(df, filename):
    path = os.path.join(OUTPUT_TABLES_DIR, filename)
    df.to_csv(path, index=False)
    return path

def save_figure(filename):
    path = os.path.join(OUTPUT_FIGURES_DIR, filename)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    return path

def write_summary(lines):
    with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

DATASET_PATH = resolve_dataset_path(DATASET_PATH)

print("Dataset path:", DATASET_PATH)
print("Output directory:", OUTPUT_BASE_DIR)

## Load the public validation dataset

This cell reads the public `summer_peaks.csv` dataset from Google Drive and prints the first rows. The purpose is to validate that the notebook is running on the intended public dataset used in the main paper experiment.

In [ ]:
print("[1/10] Loading dataset...")
df = pd.read_csv(DATASET_PATH)
print("Shape:", df.shape)
display(df.head())

## Profile the dataset

This cell creates a compact profile of each column: type, missing values, and number of unique values. The resulting table is also saved for the paper and repository.

In [ ]:
print("[2/10] Profiling dataset...")
profile_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing": [int(df[c].isna().sum()) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=False)) for c in df.columns],
})
dataset_profile_path = save_table(profile_df, "dataset_profile.csv")
display(profile_df)
print("Saved:", dataset_profile_path)

## Preprocess calendar and operational features

This cell converts the date column, creates compact calendar features, and prepares a clean tabular representation for the prediction pipeline.

In [ ]:
print("[3/10] Preprocessing...")
df["day"] = pd.to_datetime(df["day"])
df["month"] = df["day"].dt.month
df["day_of_month"] = df["day"].dt.day
df["day_of_week"] = df["day"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

display(df.head())

## Build a synthetic urban district

This cell expands the original profile into a small heterogeneous building cluster. The cluster preserves the spirit of the public data while enabling a district-level coordination experiment with multiple buildings.

In [ ]:
print("[4/10] Building multi-building urban dataset...")

building_factors = {
    "B1": {"energy_scale": 0.95, "peak_power_scale": 0.96, "activity_shift": -0.02},
    "B2": {"energy_scale": 1.00, "peak_power_scale": 1.00, "activity_shift":  0.00},
    "B3": {"energy_scale": 1.06, "peak_power_scale": 1.04, "activity_shift":  0.03},
    "B4": {"energy_scale": 0.90, "peak_power_scale": 0.92, "activity_shift": -0.04},
    "B5": {"energy_scale": 1.10, "peak_power_scale": 1.08, "activity_shift":  0.05},
}

urban_frames = []
for bid, factors in building_factors.items():
    tmp = df.copy()
    tmp["building_id"] = bid
    tmp["energy"] = tmp["energy"] * factors["energy_scale"]
    tmp["peak_power"] = tmp["peak_power"] * factors["peak_power_scale"]
    tmp["activity"] = np.clip(tmp["activity"] + factors["activity_shift"], 0, None)
    urban_frames.append(tmp)

urban_df = pd.concat(urban_frames, ignore_index=True)
urban_path = save_table(urban_df.head(1000), "urban_cluster_sample.csv")

print("Urban dataset shape:", urban_df.shape)
print("Buildings:", urban_df["building_id"].nunique())
print("Saved sample:", urban_path)
display(urban_df.head())

## Train the demand prediction model

This cell trains a Ridge-regression pipeline with numeric preprocessing and one-hot encoding for building identity. Prediction metrics are saved for the paper.

In [ ]:
print("[5/10] Training prediction model...")

target_col = "energy"
numeric_cols = [
    "wd", "T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10",
    "peak_power", "peak_duration", "peak_intensity", "is_peak", "activity",
    "month", "day_of_month", "day_of_week", "is_weekend"
]
categorical_cols = ["building_id"]

X = urban_df[numeric_cols + categorical_cols]
y = urban_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.40, random_state=SEED
)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_cols),
])

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0)),
])

ridge_model.fit(X_train, y_train)
pred_test = ridge_model.predict(X_test)

prediction_metrics = {
    "R2": r2_score(y_test, pred_test),
    "MAE": mean_absolute_error(y_test, pred_test),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred_test)),
    "ExplainedVariance": explained_variance_score(y_test, pred_test),
}

prediction_metrics_df = pd.DataFrame([prediction_metrics])
prediction_metrics_path = save_table(prediction_metrics_df, "ridge_metrics_temporal_optimization.csv")
display(prediction_metrics_df)
print("Saved:", prediction_metrics_path)

## Aggregate district demand and define operating thresholds

This cell predicts building demand, aggregates it at district level, and computes the reference load, grid-capacity proxy, and stress limit used later by the optimizer.

In [ ]:
print("[6/10] Aggregating district demand...")

urban_pred = ridge_model.predict(urban_df[numeric_cols + categorical_cols])

daily_cluster = urban_df[["day", "building_id", "energy"]].copy()
daily_cluster["predicted_energy"] = urban_pred

district_df = daily_cluster.groupby("day", as_index=False).agg(
    district_actual_energy=("energy", "sum"),
    district_predicted_energy=("predicted_energy", "sum"),
)

grid_capacity_proxy = district_df["district_predicted_energy"].quantile(0.80)
L_ref = 0.85 * grid_capacity_proxy

district_df["district_stress_index"] = district_df["district_predicted_energy"] / grid_capacity_proxy
district_path = save_table(district_df, "daily_district_metrics_before_sparse_budget_optimization.csv")

print("Grid capacity proxy:", round(grid_capacity_proxy, 4))
print("Reference load L_ref:", round(L_ref, 4))
print("Saved:", district_path)
display(district_df.head(10))

## Optimization configuration

This cell defines the coordination hyperparameters in one dictionary. Grouping the settings here makes the notebook easier to read and makes the ablation study reproducible.

In [ ]:
print("[7/10] Defining optimization configuration...")

base_config = {
    "max_sender_shift_ratio": 0.12,
    "max_receiver_increase_ratio": 0.10,
    "storage_capacity_ratio": 0.10,
    "storage_rate_ratio": 0.05,
    "storage_initial_ratio": 0.50,
    "eta_c": 0.95,
    "eta_d": 0.95,
    "stress_ceiling": 0.92,
    "duration_threshold": 0.88,
    "rolling_window_days": 14,
    "rolling_window_overload_mass_budget_ratio": 0.02,
    "total_overload_mass_budget_ratio": 0.50,
    "curtailment_cap_ratio": 0.002,
    "daily_curtailment_cap_ratio": 0.005,
    "overload_deadband_ratio": 0.01,
    "violation_day_budget_ratio": 0.40,
    "weights": {
        "peak": 30.0,
        "excess": 25.0,
        "variance": 0.6,
        "shift": 2.0,
        "storage": 0.6,
        "curtail": 20000.0,
        "realism": 1.0,
        "fairness": 2.0,
        "stress_slack": 5000.0,
        "rolling_slack": 5000.0,
        "total_overload_slack": 5000.0,
        "conservation_slack": 5000.0,
        "terminal_soc_slack": 5000.0,
        "overload_l2": 400.0,
        "overload_l1": 400.0,
        "violation_activation": 5000.0,
    },
}
base_config

## Optimization helpers

This cell imports CVXPY and defines the reusable coordination solver. The solver returns both the coordinated time series and a rich metric dictionary. The thresholded violation-day metric is treated as the main indicator of significant overload persistence.

In [ ]:
print("[8/10] Loading optimizer and defining solver...")
import cvxpy as cp

def solve_coordination(baseline_load, grid_capacity_proxy, L_ref, config, regime_name="sparse_budget"):
    start_time = time.time()
    n = len(baseline_load)
    weights = config["weights"]

    S_max = config["storage_capacity_ratio"] * baseline_load.max()
    C_max = config["storage_rate_ratio"] * baseline_load.max()
    D_max = config["storage_rate_ratio"] * baseline_load.max()
    initial_soc = config["storage_initial_ratio"] * S_max

    native_shift_cap = config["max_sender_shift_ratio"] * baseline_load
    native_receive_cap = config["max_receiver_increase_ratio"] * baseline_load
    daily_curtail_cap = config["daily_curtailment_cap_ratio"] * baseline_load
    total_curtail_cap = config["curtailment_cap_ratio"] * baseline_load.sum()

    rolling_budget = (
        config["rolling_window_overload_mass_budget_ratio"]
        * baseline_load.sum() / n * config["rolling_window_days"]
    )
    total_overload_budget = config["total_overload_mass_budget_ratio"] * baseline_load.sum()

    smooth_target = 0.84 * grid_capacity_proxy
    overload_threshold = config["duration_threshold"] * grid_capacity_proxy
    overload_deadband = config["overload_deadband_ratio"] * overload_threshold
    violation_threshold = overload_threshold + overload_deadband
    K_max = max(1, int(np.ceil(config["violation_day_budget_ratio"] * n)))
    big_m_overload = max(
        1.0,
        float(
            baseline_load.max()
            * (1.0 + config["max_sender_shift_ratio"] + config["max_receiver_increase_ratio"] + config["storage_rate_ratio"])
        ),
    )
    min_active_overload = max(1.0, overload_deadband)

    shift_out = cp.Variable(n, nonneg=True)
    shift_in = cp.Variable(n, nonneg=True)
    charge = cp.Variable(n, nonneg=True)
    discharge = cp.Variable(n, nonneg=True)
    soc = cp.Variable(n + 1)
    curtail = cp.Variable(n, nonneg=True)
    peak = cp.Variable(nonneg=True)
    excess = cp.Variable(n, nonneg=True)
    overload_mass = cp.Variable(n, nonneg=True)
    violation_activation = cp.Variable(n)

    stress_slack = cp.Variable(n, nonneg=True)
    rolling_slack = cp.Variable(max(0, n - config["rolling_window_days"] + 1), nonneg=True)
    total_overload_slack = cp.Variable(nonneg=True)
    conservation_slack = cp.Variable(nonneg=True)
    terminal_soc_slack = cp.Variable(nonneg=True)

    load_after = baseline_load - shift_out + shift_in + charge - discharge - curtail

    constraints = [soc[0] == initial_soc]
    for t in range(n):
        constraints += [
            shift_out[t] <= native_shift_cap[t],
            shift_in[t] <= native_receive_cap[t],
            charge[t] <= C_max,
            discharge[t] <= D_max,
            curtail[t] <= daily_curtail_cap[t],
            soc[t + 1] == soc[t] + config["eta_c"] * charge[t] - discharge[t] / config["eta_d"],
            soc[t + 1] >= 0,
            soc[t + 1] <= S_max,
            peak >= load_after[t],
            excess[t] >= load_after[t] - L_ref,
            overload_mass[t] >= load_after[t] - violation_threshold,
            load_after[t] <= config["stress_ceiling"] * grid_capacity_proxy + stress_slack[t],
            violation_activation[t] >= 0,
            violation_activation[t] <= 1,
            overload_mass[t] <= big_m_overload * violation_activation[t],
            overload_mass[t] >= min_active_overload * violation_activation[t],
        ]

    constraints += [
        cp.sum(shift_out) - cp.sum(shift_in) <= conservation_slack,
        cp.sum(shift_in) - cp.sum(shift_out) <= conservation_slack,
        cp.sum(curtail) <= total_curtail_cap,
        cp.sum(overload_mass) <= total_overload_budget + total_overload_slack,
        cp.sum(violation_activation) <= K_max,
        soc[n] - initial_soc <= terminal_soc_slack,
        initial_soc - soc[n] <= terminal_soc_slack,
    ]

    for start in range(0, n - config["rolling_window_days"] + 1):
        constraints += [
            cp.sum(overload_mass[start:start + config["rolling_window_days"]]) <= rolling_budget + rolling_slack[start]
        ]

    max_total_rolling_slack = 0.15 * total_overload_budget
    max_total_stress_slack = 0.01 * grid_capacity_proxy * n
    constraints += [
        cp.sum(rolling_slack) <= max_total_rolling_slack,
        cp.sum(stress_slack) <= max_total_stress_slack,
    ]

    intervention = shift_out + shift_in + charge + discharge + curtail
    mean_intervention = cp.sum(intervention) / n

    objective = cp.Minimize(
        weights["peak"] * peak
        + weights["excess"] * cp.sum(excess)
        + weights["variance"] * cp.sum_squares(load_after - np.mean(baseline_load))
        + weights["realism"] * cp.sum_squares(load_after - smooth_target)
        + weights["shift"] * (cp.sum(shift_out) + cp.sum(shift_in))
        + weights["storage"] * (cp.sum(charge) + cp.sum(discharge))
        + weights["curtail"] * cp.sum(curtail)
        + weights["fairness"] * cp.sum_squares(intervention - mean_intervention) / n
        + weights["stress_slack"] * cp.sum(stress_slack)
        + weights["rolling_slack"] * cp.sum(rolling_slack)
        + weights["total_overload_slack"] * total_overload_slack
        + weights["conservation_slack"] * conservation_slack
        + weights["terminal_soc_slack"] * terminal_soc_slack
        + weights["overload_l2"] * cp.sum_squares(overload_mass)
        + weights["overload_l1"] * cp.sum(overload_mass)
        + weights["violation_activation"] * cp.sum(violation_activation)
    )

    problem = cp.Problem(objective, constraints)

    solver_used = None
    solution_status = None
    last_error = None
    solver_sequence = [
        ("SCS", cp.SCS, {"verbose": False, "max_iters": 30000}),
        ("CLARABEL", cp.CLARABEL, {"verbose": False}),
        ("OSQP", cp.OSQP, {"verbose": False}),
    ]

    for solver_name, solver_obj, solver_kwargs in solver_sequence:
        try:
            problem.solve(solver=solver_obj, **solver_kwargs)
            solver_used = solver_name
            solution_status = problem.status
            if solution_status in ["optimal", "optimal_inaccurate"]:
                break
        except Exception as exc:
            last_error = repr(exc)

    if solution_status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimization failed. Last status: {solution_status}. Last error: {last_error}")

    results_df = pd.DataFrame({
        "day": district_df["day"],
        "load_before": baseline_load,
        "load_after": load_after.value,
        "shift_out": shift_out.value,
        "shift_in": shift_in.value,
        "charge": charge.value,
        "discharge": discharge.value,
        "curtail": curtail.value,
        "soc": soc.value[1:],
        "excess": excess.value,
        "overload_mass": overload_mass.value,
        "violation_activation": violation_activation.value,
        "stress_slack": stress_slack.value,
    })

    overload_before = baseline_load > L_ref
    overload_after = results_df["load_after"].to_numpy() > L_ref
    thresholded_after = results_df["load_after"].to_numpy() > violation_threshold
    stress_limit = config["stress_ceiling"] * grid_capacity_proxy

    metrics = {
        "regime": regime_name,
        "solver_used": solver_used,
        "solution_status": solution_status,
        "runtime_sec": time.time() - start_time,
        "peak_before": float(baseline_load.max()),
        "peak_after": float(results_df["load_after"].max()),
        "peak_reduction_ratio": float((baseline_load.max() - results_df["load_after"].max()) / baseline_load.max()),
        "excess_sum_before": float(np.maximum(baseline_load - L_ref, 0).sum()),
        "excess_sum_after": float(np.maximum(results_df["load_after"].to_numpy() - L_ref, 0).sum()),
        "excess_reduction_ratio": float(
            (np.maximum(baseline_load - L_ref, 0).sum() - np.maximum(results_df["load_after"].to_numpy() - L_ref, 0).sum())
            / max(np.maximum(baseline_load - L_ref, 0).sum(), 1e-9)
        ),
        "variance_before": float(np.var(baseline_load)),
        "variance_after": float(np.var(results_df["load_after"].to_numpy())),
        "variance_reduction_ratio": float((np.var(baseline_load) - np.var(results_df["load_after"].to_numpy())) / max(np.var(baseline_load), 1e-9)),
        "overload_days_before": int(overload_before.sum()),
        "overload_days_after": int(overload_after.sum()),
        "violation_days_after_thresholded": int(thresholded_after.sum()),
        "stress_ceiling_days_after": int((results_df["load_after"].to_numpy() > stress_limit).sum()),
        "duration_threshold_days_after": int((results_df["load_after"].to_numpy() > overload_threshold).sum()),
        "total_shifted_out_energy": float(results_df["shift_out"].sum()),
        "total_shifted_in_energy": float(results_df["shift_in"].sum()),
        "total_storage_charge": float(results_df["charge"].sum()),
        "total_storage_discharge": float(results_df["discharge"].sum()),
        "total_curtailment": float(results_df["curtail"].sum()),
        "curtailment_share_of_total_load": float(results_df["curtail"].sum() / baseline_load.sum()),
        "intervention_std": float((results_df["shift_out"] + results_df["shift_in"] + results_df["charge"] + results_df["discharge"] + results_df["curtail"]).std()),
        "total_stress_slack": float(stress_slack.value.sum()),
        "total_rolling_slack": float(rolling_slack.value.sum()) if rolling_slack.size else 0.0,
        "total_overload_budget_slack": float(total_overload_slack.value),
        "conservation_slack": float(conservation_slack.value),
        "terminal_soc_slack": float(terminal_soc_slack.value),
        "objective_overload_l2_component": float(cp.sum_squares(overload_mass).value),
        "objective_overload_l1_component": float(cp.sum(overload_mass).value),
        "objective_violation_activation_component": float(cp.sum(violation_activation).value),
        "violation_activation_budget_used": float(cp.sum(violation_activation).value),
        "violation_activation_budget_max": int(K_max),
        "violation_threshold": float(violation_threshold),
        "stress_limit": float(stress_limit),
        "reference_load": float(L_ref),
        "grid_capacity_proxy": float(grid_capacity_proxy),
    }

    return results_df, metrics

## Main optimization run

This cell runs the temporally sparse, storage-augmented coordination model on the district baseline load and stores the main outputs used in the rest of the notebook.

In [ ]:
print("[9/10] Running main optimization...")
baseline_load = district_df["district_predicted_energy"].to_numpy(dtype=float)

results_df, optimization_metrics = solve_coordination(
    baseline_load=baseline_load,
    grid_capacity_proxy=grid_capacity_proxy,
    L_ref=L_ref,
    config=base_config,
    regime_name="temporal_sparse_budget",
)

optimization_metrics_df = pd.DataFrame([optimization_metrics])
optimization_metrics_path = save_table(optimization_metrics_df, "optimization_metrics_temporal_sparse_budget.csv")
optimization_results_path = save_table(results_df, "optimization_results_temporal_sparse_budget.csv")

display(optimization_metrics_df.T)
print("Saved:", optimization_metrics_path)
print("Saved:", optimization_results_path)

## Core paper tables

This cell exports compact tables for the manuscript, including headline metrics and the most stressed days before and after coordination.

In [ ]:
print("[10/10] Building core paper tables...")

paper_metrics_df = pd.DataFrame([
    {"metric": "Peak reduction ratio", "value": optimization_metrics["peak_reduction_ratio"]},
    {"metric": "Excess reduction ratio", "value": optimization_metrics["excess_reduction_ratio"]},
    {"metric": "Variance reduction ratio", "value": optimization_metrics["variance_reduction_ratio"]},
    {"metric": "Curtailment share", "value": optimization_metrics["curtailment_share_of_total_load"]},
    {"metric": "Raw overload days after", "value": optimization_metrics["overload_days_after"]},
    {"metric": "Thresholded violation days after", "value": optimization_metrics["violation_days_after_thresholded"]},
    {"metric": "Stress ceiling days after", "value": optimization_metrics["stress_ceiling_days_after"]},
    {"metric": "Activation budget used", "value": optimization_metrics["violation_activation_budget_used"]},
])

top_before = pd.DataFrame({
    "day": district_df["day"],
    "load_before": baseline_load,
}).sort_values("load_before", ascending=False).head(15)

top_after = results_df[["day", "load_after", "overload_mass", "violation_activation"]].sort_values(
    "load_after", ascending=False
).head(15)

paper_metrics_path = save_table(paper_metrics_df, "paper_metrics_table.csv")
top_before_path = save_table(top_before, "top_stress_days_before.csv")
top_after_path = save_table(top_after, "top_stress_days_after.csv")

display(paper_metrics_df)
display(top_before)
display(top_after)

print("Saved:", paper_metrics_path)
print("Saved:", top_before_path)
print("Saved:", top_after_path)

## Figures for the paper

This cell exports the main figures needed for the paper and repository: district load evolution, storage behavior, intervention components, excess profile, and metric comparison.

In [ ]:
print("Exporting figures...")

# 1. District load before vs after
plt.figure(figsize=(11, 5))
plt.plot(results_df["day"], results_df["load_before"], label="Before")
plt.plot(results_df["day"], results_df["load_after"], label="After")
plt.axhline(L_ref, linestyle="--", label="L_ref")
plt.axhline(optimization_metrics["violation_threshold"], linestyle=":", label="Violation threshold")
plt.xticks(rotation=45)
plt.ylabel("District predicted energy")
plt.title("District load before vs after optimization")
plt.legend()
plt.tight_layout()
district_figure_path = save_figure("district_load_sparse_budget.png")

# 2. Storage SOC
plt.figure(figsize=(11, 4))
plt.plot(results_df["day"], results_df["soc"])
plt.xticks(rotation=45)
plt.ylabel("State of charge")
plt.title("Storage state of charge")
plt.tight_layout()
soc_figure_path = save_figure("storage_soc_sparse_budget.png")

# 3. Intervention components
plt.figure(figsize=(11, 5))
plt.plot(results_df["day"], results_df["shift_out"], label="Shift out")
plt.plot(results_df["day"], results_df["shift_in"], label="Shift in")
plt.plot(results_df["day"], results_df["charge"], label="Charge")
plt.plot(results_df["day"], results_df["discharge"], label="Discharge")
plt.plot(results_df["day"], results_df["curtail"], label="Curtail")
plt.xticks(rotation=45)
plt.ylabel("Energy")
plt.title("Intervention components over time")
plt.legend(ncol=3)
plt.tight_layout()
intervention_figure_path = save_figure("intervention_components_sparse_budget.png")

# 4. Exceedance view
before_excess = np.maximum(results_df["load_before"].to_numpy() - L_ref, 0)
after_excess = np.maximum(results_df["load_after"].to_numpy() - L_ref, 0)
plt.figure(figsize=(11, 4))
plt.plot(results_df["day"], before_excess, label="Before excess")
plt.plot(results_df["day"], after_excess, label="After excess")
plt.xticks(rotation=45)
plt.ylabel("Excess above L_ref")
plt.title("Excess-load profile before vs after")
plt.legend()
plt.tight_layout()
excess_figure_path = save_figure("excess_profile_sparse_budget.png")

# 5. Metric comparison
comparison_df = pd.DataFrame({
    "metric": ["Peak", "Excess sum", "Variance", "Raw overload days", "Thresholded violation days"],
    "before": [
        optimization_metrics["peak_before"],
        optimization_metrics["excess_sum_before"],
        optimization_metrics["variance_before"],
        optimization_metrics["overload_days_before"],
        optimization_metrics["overload_days_before"],
    ],
    "after": [
        optimization_metrics["peak_after"],
        optimization_metrics["excess_sum_after"],
        optimization_metrics["variance_after"],
        optimization_metrics["overload_days_after"],
        optimization_metrics["violation_days_after_thresholded"],
    ],
})

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison_df))
width = 0.35
ax.bar(x - width/2, comparison_df["before"], width, label="Before")
ax.bar(x + width/2, comparison_df["after"], width, label="After")
ax.set_xticks(x)
ax.set_xticklabels(comparison_df["metric"], rotation=20)
ax.set_title("Before/after comparison of key optimization metrics")
ax.legend()
plt.tight_layout()
comparison_figure_path = save_figure("metric_comparison_sparse_budget.png")

comparison_table_path = save_table(comparison_df, "before_after_metric_comparison.csv")

print("Saved:", district_figure_path)
print("Saved:", soc_figure_path)
print("Saved:", intervention_figure_path)
print("Saved:", excess_figure_path)
print("Saved:", comparison_figure_path)
print("Saved:", comparison_table_path)

## Optional ablation study

Ablation is useful for the paper, but it can be slow because it reruns the optimization for multiple regimes.
By default, the notebook keeps ablation **disabled** so the main workflow finishes quickly and predictably.

If you want to run it, set:
- `ABLATION_ENABLED = True`

If you want a lighter ablation, keep:
- `ABLATION_FAST_MODE = True`

This will reduce solver effort for the auxiliary regimes while preserving the main experiment unchanged.


In [ ]:
print("Ablation study")

# Keep ablation optional so the main notebook remains fast and GitHub-friendly.
ABLATION_ENABLED = False
ABLATION_FAST_MODE = True

ablation_df = pd.DataFrame()
ablation_path = None
ablation_figure_path = None

if not ABLATION_ENABLED:
    print("Ablation skipped. Set ABLATION_ENABLED = True to run it.")
else:
    print("Running ablation study...")

    def make_ablation_config(cfg, *, deadband_ratio, budget_ratio, activation_weight):
        new_cfg = {
            **cfg,
            "overload_deadband_ratio": deadband_ratio,
            "violation_day_budget_ratio": budget_ratio,
            "weights": {**cfg["weights"], "violation_activation": activation_weight},
        }
        if ABLATION_FAST_MODE:
            new_cfg = {
                **new_cfg,
                "solver_sequence": ["SCS"],
                "solver_settings": {
                    **new_cfg["solver_settings"],
                    "SCS": {
                        **new_cfg["solver_settings"]["SCS"],
                        "max_iters": min(2500, new_cfg["solver_settings"]["SCS"].get("max_iters", 6000)),
                        "eps_abs": max(1e-3, new_cfg["solver_settings"]["SCS"].get("eps_abs", 1e-3)),
                        "eps_rel": max(1e-3, new_cfg["solver_settings"]["SCS"].get("eps_rel", 1e-3)),
                        "verbose": False,
                    },
                },
            }
        return new_cfg

    ablation_configs = {
        "baseline_convex": make_ablation_config(
            base_config,
            deadband_ratio=0.00,
            budget_ratio=1.00,
            activation_weight=0.0,
        ),
        "threshold_only": make_ablation_config(
            base_config,
            deadband_ratio=0.01,
            budget_ratio=1.00,
            activation_weight=0.0,
        ),
        "sparse_budget_full": make_ablation_config(
            base_config,
            deadband_ratio=base_config["overload_deadband_ratio"],
            budget_ratio=base_config["violation_day_budget_ratio"],
            activation_weight=base_config["weights"]["violation_activation"],
        ),
    }

    ablation_rows = []
    total_regimes = len(ablation_configs)

    for idx, (name, cfg) in enumerate(ablation_configs.items(), start=1):
        print(f"  [{idx}/{total_regimes}] Running ablation regime: {name}")
        _, m = solve_coordination(
            baseline_load=baseline_load,
            grid_capacity_proxy=grid_capacity_proxy,
            L_ref=L_ref,
            config=cfg,
            regime_name=name,
        )
        ablation_rows.append(m)
        print(
            f"      done: peak_reduction={m['peak_reduction_ratio']:.4f}, "
            f"excess_reduction={m['excess_reduction_ratio']:.4f}, "
            f"thresholded_violation_days={int(m['violation_days_after_thresholded'])}"
        )

    ablation_df = pd.DataFrame(ablation_rows)
    ablation_path = save_table(ablation_df, "ablation_study_sparse_budget.csv")
    display(ablation_df[[
        "regime",
        "peak_reduction_ratio",
        "excess_reduction_ratio",
        "curtailment_share_of_total_load",
        "overload_days_after",
        "violation_days_after_thresholded",
        "stress_ceiling_days_after",
        "violation_activation_budget_used"
    ]])

    plt.figure(figsize=(9, 5))
    x = np.arange(len(ablation_df))
    width = 0.35
    plt.bar(x - width/2, ablation_df["peak_reduction_ratio"], width=width, label="Peak reduction")
    plt.bar(x + width/2, ablation_df["excess_reduction_ratio"], width=width, label="Excess reduction")
    plt.xticks(x, ablation_df["regime"], rotation=15)
    plt.ylabel("Ratio")
    plt.title("Ablation performance across regimes")
    plt.legend()
    plt.tight_layout()
    ablation_figure_path = save_figure("ablation_performance_sparse_budget.png")
    plt.show()


## Rich summary report

This final cell writes a detailed `outputs_summary.txt` that records prediction quality, optimization metrics, thresholded stress behavior, saved artifacts, and ablation results.

In [ ]:
print("Writing rich summary report...")

summary_lines = [
    "Urban Energy Coordination",
    "========================",
    "",
    "Notebook overview",
    "-----------------",
    "Key ideas:",
    "- peak reduction",
    "- excess reduction",
    "- temporal sparsity control",
    "- thresholded violation metric",
    "- budgeted activation constraint",
    "",
    "Prediction stage",
    "----------------",
]
for key, value in prediction_metrics.items():
    summary_lines.append(f"{key}: {value:.6f}")

summary_lines += [
    "",
    "Main optimization stage",
    "-----------------------",
]
for key, value in optimization_metrics.items():
    if isinstance(value, float):
        summary_lines.append(f"{key}: {value:.6f}")
    else:
        summary_lines.append(f"{key}: {value}")

summary_lines += [
    "",
    "Interpretation notes",
    "--------------------",
    f"Raw overload days after: {optimization_metrics['overload_days_after']}",
    f"Thresholded violation days after: {optimization_metrics['violation_days_after_thresholded']}",
    "The thresholded metric is the primary indicator of significant overload persistence.",
    "Raw overload-day counts can remain high because convex formulations may leave many negligible residual violations.",
    "",
    "Pass/fail checks",
    "----------------",
    f"[{'PASS' if optimization_metrics['peak_reduction_ratio'] > 0.10 else 'FAIL'}] Peak reduction > 10%",
    f"[{'PASS' if optimization_metrics['curtailment_share_of_total_load'] < 0.005 else 'FAIL'}] Curtailment share < 0.5%",
    f"[{'PASS' if optimization_metrics['violation_days_after_thresholded'] < optimization_metrics['overload_days_before'] else 'FAIL'}] Significant violation days reduced",
    f"[{'PASS' if optimization_metrics['stress_ceiling_days_after'] <= 10 else 'FAIL'}] Stress ceiling days after <= 10",
    "",
    "Ablation study",
    "--------------",
]
for _, row in ablation_df.iterrows():
    summary_lines.append(
        f"{row['regime']}: peak_reduction_ratio={row['peak_reduction_ratio']:.4f}, "
        f"excess_reduction_ratio={row['excess_reduction_ratio']:.4f}, "
        f"thresholded_violation_days={int(row['violation_days_after_thresholded'])}, "
        f"curtailment_share={row['curtailment_share_of_total_load']:.6f}"
    )

if ablation_df.empty:
    summary_lines += [
        "",
        "Ablation study",
        "--------------",
        "Ablation skipped in default notebook mode.",
    ]
else:
    summary_lines += [
        "",
        "Ablation study",
        "--------------",
    ]
    for _, row in ablation_df.iterrows():
        summary_lines.append(
            f"{row['regime']}: peak_reduction_ratio={row['peak_reduction_ratio']:.4f}, "
            f"excess_reduction_ratio={row['excess_reduction_ratio']:.4f}, "
            f"thresholded_violation_days={int(row['violation_days_after_thresholded'])}, "
            f"curtailment_share={row['curtailment_share_of_total_load']:.6f}"
        )

saved_tables = [
    prediction_metrics_path,
    optimization_metrics_path,
    optimization_results_path,
    paper_metrics_path,
    top_before_path,
    top_after_path,
    comparison_table_path,
]
if ablation_path is not None:
    saved_tables.append(ablation_path)

saved_figures = [
    district_figure_path,
    soc_figure_path,
    intervention_figure_path,
    excess_figure_path,
    comparison_figure_path,
]
if ablation_figure_path is not None:
    saved_figures.append(ablation_figure_path)

summary_lines += [
    "",
    "Saved tables",
    "------------",
]
summary_lines += saved_tables
summary_lines += [
    "",
    "Saved figures",
    "-------------",
]
summary_lines += saved_figures

write_summary(summary_lines)
print("Summary written to:", SUMMARY_PATH)
print("\n".join(summary_lines[:40]))